# 05 · Incremental load with Delta MERGE
batch_002 contains new leases and tenants, renewals, late payments, re-sent unchanged rows and
a few invalid rows. We load it, inspect what changed, then run it again to prove the load is
idempotent.

In [ ]:
%pip install -q pyyaml python-dateutil

In [ ]:
# Setup: make the repo's `src` package importable and select the Databricks environment.
# The notebook runs from <repo>/notebooks inside a Databricks Git folder.
import os, sys
repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.environ["LAKEHOUSE_ENV"] = "databricks"

dbutils.widgets.text("catalog", "workspace")          # Free Edition default; production: real_estate
catalog = dbutils.widgets.get("catalog")
overrides = {"environments": {"databricks": {"catalog": catalog,
                                             "raw_volume_root": f"/Volumes/{catalog}/raw/landing"}}}

from src.pipeline import build_lakehouse
lh, cfg = build_lakehouse(env="databricks", overrides=overrides)
print(f"catalog={cfg.catalog}  as_of={cfg.reporting_as_of}  batches={list(cfg.batches)}")

In [ ]:
def counts():
    return {t: spark.table(f"{catalog}.silver.{t}").count() for t in ["tenants", "leases", "payments", "valuations"]}
before = counts(); before

In [ ]:
from src.pipeline import run_bronze, run_silver, run_gold
run_bronze(lh, cfg, "batch_002")
run_silver(lh, cfg, "batch_002")
after = counts()
{t: (before[t], after[t]) for t in before}

### MERGE updated renewed leases in place (same lease_id, new end date and rent)

In [ ]:
display(spark.table(f"{catalog}.silver.leases").filter("batch_id = 'batch_002'").orderBy("lease_id"))

### SCD Type 2: tenants with a contact change now have two versions

In [ ]:
display(spark.sql(f"""
SELECT tenant_id, email, phone, effective_from, effective_to, is_current
FROM {catalog}.silver.tenants
WHERE tenant_id IN (SELECT tenant_id FROM {catalog}.silver.tenants GROUP BY tenant_id HAVING COUNT(*) > 1)
ORDER BY tenant_id, effective_from"""))

### Delta history shows the MERGE operations

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {catalog}.silver.leases").select("version", "timestamp", "operation", "operationMetrics"))

### Run batch_002 again: nothing changes

In [ ]:
run_bronze(lh, cfg, "batch_002")
run_silver(lh, cfg, "batch_002")
assert counts() == after, "re-running a batch must not change Silver"
print("idempotent:", counts())

In [ ]:
run_gold(lh, cfg)
display(spark.table(f"{catalog}.gold.portfolio_summary"))